# Explore table freshness
Examines Delta table history to find when each table last changed, as groundwork for freshness checks.

In [0]:
%sql
-- See every recorded change to the customer dev table
describe history validation_sweep.dev.customer;

In [0]:
%sql
-- Compare the history of a table rebuilt on every run
describe history validation_sweep.dev.customer_dupes;

In [0]:
%sql
-- Create a small table, then make three kinds of changes to it
create or replace table validation_sweep.dev.freshness_demo as
select * from samples.tpch_sf1.region;

insert into validation_sweep.dev.freshness_demo
select * from samples.tpch_sf1.region where r_regionkey = 0;

optimize validation_sweep.dev.freshness_demo;

describe history validation_sweep.dev.freshness_demo;

In [0]:
# Find the most recent change that wrote data, ignoring maintenance such as OPTIMIZE
from pyspark.sql import functions as F

history = spark.sql("describe history validation_sweep.dev.freshness_demo")

# Keep only changes that wrote rows; OPTIMIZE and other maintenance
# have no numOutputRows, so the comparison is null and they're excluded
data_changes = history.where(
    F.col("operationMetrics").getItem("numOutputRows").cast("long") > 0
)

display(data_changes.select("version", "timestamp", "operation"))

last_data_change = data_changes.agg(F.max("timestamp")).first()[0]
print(f"Last data change: {last_data_change}")

In [0]:
%sql
drop table validation_sweep.dev.freshness_demo;